# Assignment 5: Comparative Binary Classification with XGBoost

Replace `lastname_firstname` with your own name before submission.


## 1. Problem Definition
- Target variable: `Bankrupt?`
- Positive class: `1`, meaning the company went bankrupt.
- Missing a bankrupt company is worse than incorrectly flagging a healthy company.
- The dataset is imbalanced, so accuracy is not the main metric.
- Main discrimination metric: PR-AUC.
- Calibration metric: Brier score.
- Test set is held until the final section only.


In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, roc_auc_score, brier_score_loss,
    precision_score, recall_score, fbeta_score, confusion_matrix,
    precision_recall_curve, roc_curve
)
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier
import joblib

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
DATA_URL = 'https://raw.githubusercontent.com/bipin-a/ml-workshop/main/training/data.csv'
DATA_DIR = Path('training'); DATA_DIR.mkdir(exist_ok=True)
DATA_PATH = DATA_DIR / 'data.csv'
ARTIFACT_DIR = Path('artifacts'); ARTIFACT_DIR.mkdir(exist_ok=True)


## 2. Load Data


In [ ]:
if not DATA_PATH.exists():
    df = pd.read_csv(DATA_URL)
    df.to_csv(DATA_PATH, index=False)
else:
    df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()


## 3. Quick EDA


In [ ]:
print('Shape:', df.shape)
print('\nTarget distribution:')
print(df['Bankrupt?'].value_counts())
print('\nTarget distribution (%):')
print((df['Bankrupt?'].value_counts(normalize=True) * 100).round(2))
print('\nMissing values:', int(df.isna().sum().sum()))
print('Duplicate rows:', int(df.duplicated().sum()))

plt.figure(figsize=(5,4))
sns.countplot(data=df, x='Bankrupt?')
plt.title('Target Distribution')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'target_distribution.png', dpi=150)
plt.show()

print(f"Observation: only {df['Bankrupt?'].mean():.2%} of rows are bankrupt, so accuracy is not the primary metric.")


## 4. Train, Validation, Test Split


In [ ]:
TARGET = 'Bankrupt?'
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15/0.85, stratify=y_temp, random_state=RANDOM_STATE
)

def bal(name, labels):
    return {
        'split': name,
        'rows': len(labels),
        'bankrupt_count': int(labels.sum()),
        'non_bankrupt_count': int((labels == 0).sum()),
        'bankrupt_rate': labels.mean()
    }

split_summary = pd.DataFrame([bal('Train', y_train), bal('Validation', y_val), bal('Test', y_test)])
split_summary


## 5. Preprocessing and Leakage Check


In [ ]:
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print('Numeric features:', len(numeric_cols))
print('Categorical features:', len(categorical_cols))
print('Potential leakage-like feature names:', [
    c for c in X_train.columns
    if 'bankrupt' in c.lower() or c.lower().strip() in ['target', 'label', 'class']
])


## 6. Feature Sets


In [ ]:
feature_set_A = numeric_cols.copy()

selector = XGBClassifier(
    n_estimators=150, max_depth=3, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9,
    eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
)
selector_X = X_train[feature_set_A].fillna(X_train[feature_set_A].median(numeric_only=True))
selector.fit(selector_X, y_train)

importance_df = pd.DataFrame({
    'feature': feature_set_A,
    'importance': selector.feature_importances_
}).sort_values('importance', ascending=False)

feature_set_B = importance_df.head(min(25, len(feature_set_A)))['feature'].tolist()

print('Feature Set A:', len(feature_set_A))
print('Feature Set B:', len(feature_set_B))
importance_df.head(10)


## 7. Reusable Evaluation Function


In [ ]:
def make_xgb(**params):
    base = dict(eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)
    base.update(params)
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', XGBClassifier(**base))
    ])

def evaluate_model(exp_id, model_name, pipe, feature_set_name, features, main_settings, threshold=0.5, notes=''):
    pipe.fit(X_train[features], y_train)
    train_p = pipe.predict_proba(X_train[features])[:, 1]
    val_p = pipe.predict_proba(X_val[features])[:, 1]
    pred = (val_p >= threshold).astype(int)

    train_pr = average_precision_score(y_train, train_p)
    val_pr = average_precision_score(y_val, val_p)

    return {
        'exp_id': exp_id,
        'model': model_name,
        'feature_set': feature_set_name,
        'main_settings': main_settings,
        'train_pr_auc': train_pr,
        'val_pr_auc': val_pr,
        'overfit_gap': train_pr - val_pr,
        'val_roc_auc': roc_auc_score(y_val, val_p),
        'val_brier': brier_score_loss(y_val, val_p),
        'threshold': threshold,
        'val_precision': precision_score(y_val, pred, zero_division=0),
        'val_recall': recall_score(y_val, pred, zero_division=0),
        'val_f1_or_f2': fbeta_score(y_val, pred, beta=2, zero_division=0),
        'selected_finalist': 'No',
        'notes': notes,
        '_pipeline': pipe,
        '_features': features,
        '_val_proba': val_p
    }


## 8. The 5 Required Experiments


In [ ]:
results = []

log_reg = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE))
])
results.append(evaluate_model(
    1, 'Logistic Regression baseline', log_reg, 'A', feature_set_A,
    'class_weight=balanced, scaled', 0.5, 'Simple baseline model.'
))

results.append(evaluate_model(
    2, 'XGBoost baseline',
    make_xgb(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=1.0, colsample_bytree=1.0),
    'A', feature_set_A, 'n=200, depth=3, lr=0.05', 0.5, 'First XGBoost model.'
))

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
results.append(evaluate_model(
    3, 'XGBoost imbalance handling',
    make_xgb(n_estimators=250, max_depth=3, learning_rate=0.05, subsample=0.9,
             colsample_bytree=0.9, scale_pos_weight=scale_pos_weight),
    'A', feature_set_A, f'scale_pos_weight={scale_pos_weight:.1f}', 0.5, 'Handles rare class.'
))

candidates = [
    dict(n_estimators=300, max_depth=2, learning_rate=0.03, subsample=0.9, colsample_bytree=0.9, reg_lambda=2.0, gamma=0.0, scale_pos_weight=scale_pos_weight),
    dict(n_estimators=250, max_depth=3, learning_rate=0.04, subsample=0.85, colsample_bytree=0.85, reg_lambda=3.0, gamma=0.1, scale_pos_weight=scale_pos_weight),
    dict(n_estimators=200, max_depth=4, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0, gamma=0.2, scale_pos_weight=scale_pos_weight),
    dict(n_estimators=150, max_depth=3, learning_rate=0.08, subsample=0.9, colsample_bytree=0.75, reg_alpha=0.1, reg_lambda=4.0, scale_pos_weight=scale_pos_weight),
]
trows = []
for i, cfg in enumerate(candidates, 1):
    p = make_xgb(**cfg)
    p.fit(X_train[feature_set_A], y_train)
    vp = p.predict_proba(X_val[feature_set_A])[:, 1]
    tp = p.predict_proba(X_train[feature_set_A])[:, 1]
    trows.append({
        'candidate': i,
        'val_pr_auc': average_precision_score(y_val, vp),
        'val_brier': brier_score_loss(y_val, vp),
        'overfit_gap': average_precision_score(y_train, tp) - average_precision_score(y_val, vp),
        'config': cfg
    })

tuning_df = pd.DataFrame(trows).sort_values(['val_pr_auc', 'val_brier'], ascending=[False, True])
display(tuning_df[['candidate','val_pr_auc','val_brier','overfit_gap']])
best_cfg = tuning_df.iloc[0]['config']

results.append(evaluate_model(
    4, 'XGBoost lightly tuned', make_xgb(**best_cfg), 'A', feature_set_A,
    'best of 4 manual configs', 0.5, 'Light validation tuning only.'
))

results.append(evaluate_model(
    5, 'XGBoost selected features', make_xgb(**best_cfg), 'B', feature_set_B,
    f'best config + top {len(feature_set_B)} features', 0.5, 'Selected feature set B.'
))

results_df = pd.DataFrame([{k:v for k,v in r.items() if not k.startswith('_')} for r in results])
best_idx = results_df.sort_values(['val_pr_auc','val_brier','overfit_gap'], ascending=[False, True, True]).index[0]
results_df.loc[best_idx, 'selected_finalist'] = 'Yes'

metric_cols = ['train_pr_auc','val_pr_auc','overfit_gap','val_roc_auc','val_brier','val_precision','val_recall','val_f1_or_f2']
results_df[metric_cols] = results_df[metric_cols].round(4)
results_df.to_csv(ARTIFACT_DIR / 'experiment_results.csv', index=False)
results_df


## 9. Final Model and Threshold Selection


In [ ]:
final_exp_id = int(results_df.loc[results_df['selected_finalist'] == 'Yes', 'exp_id'].iloc[0])
final_record = next(r for r in results if r['exp_id'] == final_exp_id)
final_model = final_record['_pipeline']
final_features = final_record['_features']
val_p = final_record['_val_proba']

threshold_rows = []
for t in np.linspace(0.01, 0.99, 99):
    pred = (val_p >= t).astype(int)
    threshold_rows.append({
        'threshold': t,
        'precision': precision_score(y_val, pred, zero_division=0),
        'recall': recall_score(y_val, pred, zero_division=0),
        'f2': fbeta_score(y_val, pred, beta=2, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_rows)
best_threshold = float(threshold_df.sort_values('f2', ascending=False).iloc[0]['threshold'])

print('Selected final experiment:', final_exp_id)
print('Selected threshold:', round(best_threshold, 2))
threshold_df.sort_values('f2', ascending=False).head(10)


## 10. Final Test Evaluation


In [ ]:
test_p = final_model.predict_proba(X_test[final_features])[:, 1]
test_pred = (test_p >= best_threshold).astype(int)

final_test_metrics = {
    'test_pr_auc': average_precision_score(y_test, test_p),
    'test_roc_auc': roc_auc_score(y_test, test_p),
    'test_brier': brier_score_loss(y_test, test_p),
    'test_precision': precision_score(y_test, test_pred, zero_division=0),
    'test_recall': recall_score(y_test, test_pred, zero_division=0),
    'test_f2': fbeta_score(y_test, test_pred, beta=2, zero_division=0),
    'final_threshold': best_threshold
}

final_test_df = pd.DataFrame([final_test_metrics]).round(4)
final_test_df.to_csv(ARTIFACT_DIR / 'final_test_metrics.csv', index=False)
final_test_df


In [ ]:
cm = confusion_matrix(y_test, test_pred)
cm_df = pd.DataFrame(cm, index=['Actual 0','Actual 1'], columns=['Predicted 0','Predicted 1'])
cm_df.to_csv(ARTIFACT_DIR / 'confusion_matrix.csv')
cm_df


## 11. Final Plots


In [ ]:
precision, recall, _ = precision_recall_curve(y_test, test_p)
plt.figure(figsize=(6,4))
plt.plot(recall, precision)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'precision_recall_curve.png', dpi=150)
plt.show()

prob_true, prob_pred = calibration_curve(y_test, test_p, n_bins=8, strategy='quantile')
plt.figure(figsize=(6,4))
plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel('Mean predicted probability')
plt.ylabel('Observed bankruptcy rate')
plt.title('Calibration Curve')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'calibration_curve.png', dpi=150)
plt.show()

fpr, tpr, _ = roc_curve(y_test, test_p)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr)
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'roc_curve.png', dpi=150)
plt.show()


## 12. Interpretability


In [ ]:
xgb_core = final_model.named_steps['model']
feature_importance = pd.DataFrame({
    'feature': final_features,
    'importance': xgb_core.feature_importances_
}).sort_values('importance', ascending=False)

feature_importance.to_csv(ARTIFACT_DIR / 'feature_importance.csv', index=False)

plt.figure(figsize=(8,5))
sns.barplot(data=feature_importance.head(15), x='importance', y='feature')
plt.title('Top Feature Importances')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'feature_importance.png', dpi=150)
plt.show()

feature_importance.head(10)


## 13. Generate Results Summary


In [ ]:
def md_table(d):
    return d.to_markdown(index=False)

summary = f'''# Assignment 5 Results Summary

## Problem Statement
- Goal: predict whether a company is likely to go bankrupt.
- Target: `Bankrupt?`; positive class: `1`.
- Business rule: missing a bankrupt company is worse than incorrectly flagging a healthy company.

## Dataset Summary
- Rows: {df.shape[0]}
- Columns: {df.shape[1]}
- Positive count: {int(y.sum())}
- Positive rate: {y.mean():.2%}
- Missing values: {int(df.isna().sum().sum())}
- Duplicate rows: {int(df.duplicated().sum())}

## Metric Choice
- The positive class is rare, so accuracy is misleading.
- PR-AUC is useful because it focuses on precision and recall for the rare class.
- Brier score measures predicted probability quality; lower is better.
- Threshold metrics like precision, recall, F1, and F2 describe one operating point only and do not fully describe calibration.

## Feature Selection
- Feature Set A used all usable numeric features.
- Feature Set B used the top {len(feature_set_B)} XGBoost importance features from the training set only.
- The test set was not used for feature selection.

## Five-Experiment Results Table
{md_table(results_df)}

## Winning Model
- Selected experiment: {final_exp_id}
- Selected model: {final_record['model']}
- Selected before using the test set.
- Chosen using validation PR-AUC, validation Brier score, and overfit gap.
- Final validation-selected threshold: {best_threshold:.2f}.

## Final Test Results
{md_table(final_test_df)}

## Confusion Matrix
{md_table(cm_df.reset_index().rename(columns={'index':'Actual'}))}

## Calibration Result
- Test Brier score: {final_test_metrics['test_brier']:.4f}
- Calibration curve saved as `artifacts/calibration_curve.png`.
- Probabilities should be interpreted carefully because bankruptcy is rare.

## Feature Importance Result
- Top feature: `{feature_importance.iloc[0]['feature']}`.
- It is the most influential according to built-in XGBoost importance.
- Limitation: importance is not the same as causation.

## Overfitting Discussion
- Overfit gap = train PR-AUC minus validation PR-AUC.
- Final model overfit gap: {float(results_df.loc[results_df['selected_finalist']=='Yes','overfit_gap'].iloc[0]):.4f}.
- Use the assignment guide: 0.00–0.05 usually fine, 0.05–0.10 watch carefully, above 0.10 needs explanation.

## AI Usage
- Used Codex/AI assistance to outline the notebook and reusable evaluation function.
- Used AI assistance to generate plotting code.
- Verified manually that the test set was not used until final evaluation.
- Checked that accuracy was not used as the primary metric.
- Watched for AI suggestions that accidentally tune on the test set.
'''
Path('results_summary.md').write_text(summary, encoding='utf-8')
joblib.dump(final_model, ARTIFACT_DIR / 'final_model.joblib')
print('Saved results_summary.md and artifacts.')
